In [ ]:
# =========================================
# Cell 1 — Imports & settings
# =========================================

import numpy as np
import pandas as pd

import pymc as pm
import arviz as az
import pytensor.tensor as pt

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# =========================================
# Cell 2 — Load datasets
# =========================================

df_latent_raw = pd.read_csv("../data/processed/latent_trunk_axes.csv")
df_clin = pd.read_excel("../data/raw/DatasetIcotMond.xlsx")

print("Latent PCs shape:", df_latent_raw.shape)
print("Clinical dataset shape:", df_clin.shape)

# HARD CHECK: same number of subjects
assert len(df_latent_raw) == len(df_clin), \
    "Latent PCs and clinical dataset are not aligned in length"

In [ ]:
# =========================================
# Cell 3 — Attach subject ID (row-wise)
# =========================================

df_latent = df_latent_raw.copy()
df_latent["ID"] = df_clin["ID"].values

print(df_latent.head())

In [ ]:
# =========================================
# Cell 4 — Prepare latent severity indicators
# =========================================

LATENT_INDICATORS = [
    "PC1_Postural",
    "PC1_Rhythmic",
    "PC1_Complexity"
]

X = df_latent[LATENT_INDICATORS].values

# Standardize
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

N, K = Xz.shape
print(f"N subjects = {N}, K indicators = {K}")

In [ ]:
# =========================================
# Cell 5 — Bayesian latent motor severity (FINAL, IDENTIFIED)
# =========================================

with pm.Model() as latent_severity_model:

    # Latent severity (standard normal)
    severity = pm.Normal(
        "severity",
        mu=0,
        sigma=1,
        shape=N
    )

    # --- IDENTIFICATION ---
    # Fixed loading for PC1_Postural
    loading_postural = pt.as_tensor_variable(1.0)

    # Free loadings for remaining indicators
    loadings_free = pm.Normal(
        "loadings_free",
        mu=0,
        sigma=1,
        shape=K - 1
    )

    loadings = pt.concatenate(
        [loading_postural[None], loadings_free]
    )

    # --- MEASUREMENT NOISE ---
    # Fix sigma for anchored indicator
    sigma_postural = pt.as_tensor_variable(0.1)

    # Estimate noise for remaining indicators
    sigma_free = pm.HalfNormal(
        "sigma_free",
        sigma=0.5,
        shape=K - 1
    )

    sigma = pt.concatenate(
        [sigma_postural[None], sigma_free]
    )

    # Expected value
    mu = severity[:, None] * loadings[None, :]

    # Likelihood
    X_obs = pm.Normal(
        "X_obs",
        mu=mu,
        sigma=sigma,
        observed=Xz
    )

    trace_latent = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        target_accept=0.95,
        random_seed=RANDOM_SEED
    )

In [ ]:
# =========================================
# Cell 6 — Targeted diagnostics (FINAL)
# =========================================

az.summary(
    trace_latent,
    var_names=["loadings_free", "sigma_free"],
    hdi_prob=0.95
)

In [ ]:
az.summary(
    trace_latent,
    var_names=["severity"],
    filter_vars="like"
).head()

In [ ]:
# =========================================
# Cell 7 — Extract latent motor severity
# =========================================

# Posterior mean per soggetto
latent_severity_mean = (
    trace_latent.posterior["severity"]
    .mean(dim=("chain", "draw"))
    .values
)

# (Opzionale) Posterior sd per incertezza individuale
latent_severity_sd = (
    trace_latent.posterior["severity"]
    .std(dim=("chain", "draw"))
    .values
)

# Aggiungi al dataframe latente
df_latent["latent_severity"] = latent_severity_mean
df_latent["latent_severity_sd"] = latent_severity_sd

# Check rapido
df_latent[["ID", "latent_severity", "latent_severity_sd"]].head()

In [ ]:
# =========================================
# Cell 8 — Save latent severity score
# =========================================

df_latent_out = df_latent[
    ["ID", "latent_severity", "latent_severity_sd"]
].copy()

df_latent_out.to_csv(
    "latent_motor_severity_score.csv",
    index=False
)

print("Saved: latent_motor_severity_score.csv")
print("Shape:", df_latent_out.shape)

In [ ]:
# =========================================
# Cell 9 — Aggregate latent severity per subject (median)
# =========================================

# df_latent contiene: PC1_* + ID + latent_severity + latent_severity_sd
assert "ID" in df_latent.columns
assert "latent_severity" in df_latent.columns
assert "latent_severity_sd" in df_latent.columns

# Check duplicati
dup_counts = df_latent["ID"].value_counts()
n_dup_subjects = (dup_counts > 1).sum()
print(f"Subjects with repeated rows: {n_dup_subjects} / {df_latent['ID'].nunique()}")
print("Top repeated IDs:\n", dup_counts.head(10))

# Aggregazione: median per soggetto
df_sev_subject = (
    df_latent.groupby("ID", as_index=False)
    .agg(
        latent_severity=("latent_severity", "median"),
        latent_severity_sd=("latent_severity_sd", "median"),
        n_obs=("latent_severity", "size"),
    )
)

print("Aggregated severity shape:", df_sev_subject.shape)
df_sev_subject.head()

In [ ]:
# =========================================
# Cell 10 — Aggregate clinical data per subject
# =========================================

needed_cols = ["ID", "Age", "Duration (years)", "target_bin"]
missing = [c for c in needed_cols if c not in df_clin.columns]
assert not missing, f"Missing in df_clin: {missing}"

# Controllo duplicati clinici
dup_clin = df_clin["ID"].value_counts()
print(f"Clinical subjects: {df_clin['ID'].nunique()} | repeated rows: {(dup_clin>1).sum()}")

df_clin_subject = (
    df_clin.groupby("ID", as_index=False)
    .agg(
        Age=("Age", "first"),
        Duration_years=("Duration (years)", "first"),
        target_bin=("target_bin", "max"),   # OR rule
    )
)

print("Aggregated clinical shape:", df_clin_subject.shape)
df_clin_subject.head()

In [ ]:
# =========================================
# Cell 11 — Build subject-level modeling dataset
# =========================================

df_subject = df_clin_subject.merge(df_sev_subject, on="ID", how="inner")

print("Subject-level dataset shape:", df_subject.shape)
print("Falls prevalence (target_bin=1):", df_subject["target_bin"].mean().round(3))

df_subject[["ID", "Age", "Duration_years", "latent_severity", "n_obs", "target_bin"]].head(10)

In [ ]:
# =========================================
# Cell 12 — Save subject-level artifact
# =========================================

df_subject.to_csv("subject_level_latent_severity_and_falls.csv", index=False)
print("Saved: subject_level_latent_severity_and_falls.csv")
print("N subjects:", len(df_subject))

In [ ]:
# =========================================
# Cell 13 — Prepare design matrix (standardize)
# =========================================

df_m = df_subject.dropna(subset=["latent_severity", "Age", "Duration_years", "target_bin"]).copy()

# Standardize predictors (mean 0, sd 1)
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(df_m[["latent_severity", "Age", "Duration_years"]])

df_m["sev_z"] = X[:, 0]
df_m["age_z"] = X[:, 1]
df_m["dur_z"] = X[:, 2]

y = df_m["target_bin"].astype(int).values
print("Modeling N:", len(df_m), "| fall rate:", y.mean().round(3))

In [ ]:
# =========================================
# Cell 14 — Bayesian logistic model: Falls ~ Severity + Age + Duration
# =========================================

with pm.Model() as falls_model:

    intercept = pm.Normal("intercept", 0, 1.5)
    beta_sev = pm.Normal("beta_sev", 0, 1)
    beta_age = pm.Normal("beta_age", 0, 1)
    beta_dur = pm.Normal("beta_dur", 0, 1)

    logit_p = (
        intercept
        + beta_sev * df_m["sev_z"].values
        + beta_age * df_m["age_z"].values
        + beta_dur * df_m["dur_z"].values
    )

    p = pm.Deterministic("p_fall", pm.math.sigmoid(logit_p))

    pm.Bernoulli("falls", p=p, observed=y)

    trace_falls = pm.sample(
        draws=3000,
        tune=3000,
        chains=4,
        target_accept=0.95,
        random_seed=RANDOM_SEED
    )

az.summary(trace_falls, var_names=["intercept", "beta_sev", "beta_age", "beta_dur"], hdi_prob=0.95)

In [ ]:
# =========================================
# Cell 15 — Risk curve: P(fall) vs severity (Age=0, Duration=0)
# =========================================

# Grid di severity in z-score
sev_grid = np.linspace(-2.5, 2.5, 101)

post = trace_falls.posterior
b0 = post["intercept"].stack(sample=("chain", "draw")).values
bs = post["beta_sev"].stack(sample=("chain", "draw")).values

# Age=0, Dur=0 (media)
logit = b0[None, :] + (sev_grid[:, None] * bs[None, :])
p_grid = 1 / (1 + np.exp(-logit))

p_mean = p_grid.mean(axis=1)
p_lo = np.quantile(p_grid, 0.025, axis=1)
p_hi = np.quantile(p_grid, 0.975, axis=1)

plt.figure()
plt.plot(sev_grid, p_mean)
plt.fill_between(sev_grid, p_lo, p_hi, alpha=0.2)
plt.xlabel("Latent motor severity (z)")
plt.ylabel("P(fall)")
plt.title("Fall risk increases with latent motor severity (Age=mean, Duration=mean)")
plt.show()

In [ ]:
# =========================================
# Cell 15 — Risk curve: P(fall) vs severity (Age=0, Duration=0)
# =========================================

import matplotlib.pyplot as plt
import numpy as np

# -------------------------------
# Nature-style global settings
# -------------------------------
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.0,
})

# -------------------------------
# Grid di severity in z-score
# -------------------------------
sev_grid = np.linspace(-2.5, 2.5, 101)

post = trace_falls.posterior
b0 = post["intercept"].stack(sample=("chain", "draw")).values
bs = post["beta_sev"].stack(sample=("chain", "draw")).values

# Age = 0, Duration = 0 (media)
logit = b0[None, :] + (sev_grid[:, None] * bs[None, :])
p_grid = 1 / (1 + np.exp(-logit))

p_mean = p_grid.mean(axis=1)
p_lo = np.quantile(p_grid, 0.025, axis=1)
p_hi = np.quantile(p_grid, 0.975, axis=1)

# -------------------------------
# Plot
# -------------------------------
fig, ax = plt.subplots(figsize=(6.5, 4.2))

# Credible interval
ax.fill_between(
    sev_grid,
    p_lo,
    p_hi,
    color="#4C72B0",
    alpha=0.20,
    linewidth=0,
    label="95% credible interval"
)

# Posterior mean
ax.plot(
    sev_grid,
    p_mean,
    color="black",
    linewidth=2.2,
    label="Posterior mean"
)

# Reference lines
ax.axvline(0, linestyle="--", linewidth=1, color="black", alpha=0.7)
ax.axhline(
    p_mean[np.argmin(np.abs(sev_grid))],
    linestyle=":",
    linewidth=1,
    color="black",
    alpha=0.7
)

# Labels
ax.set_xlabel("Latent motor severity (z-score)")
ax.set_ylabel("Probability of falling")

ax.set_xlim(-2.5, 2.5)
ax.set_ylim(0, 1)

ax.set_title(
    "Fall risk as a function of latent motor severity\n"
    "(Age and disease duration at cohort mean)",
    loc="left"
)

ax.legend(frameon=False, loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# Cell 16 — Posterior probability beta_sev > 0
# =========================================

beta_sev_samples = (
    trace_falls.posterior["beta_sev"]
    .stack(sample=("chain", "draw"))
    .values
)

p_beta_sev_pos = (beta_sev_samples > 0).mean()

print(f"P(beta_sev > 0) = {p_beta_sev_pos:.3f}")

In [ ]:
# =========================================
# Cell X — Attach UPDRS-III to subject-level dataset
# =========================================

# Controllo che la colonna esista nel dataset clinico
assert "Updrs-III" in df_clin.columns, "Updrs-III not found in df_clin"

# Aggregazione clinica includendo UPDRS
df_clin_subject_ext = (
    df_clin.groupby("ID", as_index=False)
    .agg(
        Age=("Age", "first"),
        Duration_years=("Duration (years)", "first"),
        target_bin=("target_bin", "max"),
        Updrs_III=("Updrs-III", "first"),
    )
)

# Merge con severity
df_subject_ext = df_clin_subject_ext.merge(
    df_sev_subject, on="ID", how="inner"
)

print("Extended subject-level dataset shape:", df_subject_ext.shape)
df_subject_ext.head()

In [ ]:
# =========================================
# Cell Y — Prepare design matrix with UPDRS
# =========================================

df_m_ext = df_subject_ext.dropna(
    subset=["latent_severity", "Age", "Duration_years", "Updrs_III", "target_bin"]
).copy()

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

X_ext = sc.fit_transform(
    df_m_ext[["latent_severity", "Age", "Duration_years", "Updrs_III"]]
)

df_m_ext["sev_z"] = X_ext[:, 0]
df_m_ext["age_z"] = X_ext[:, 1]
df_m_ext["dur_z"] = X_ext[:, 2]
df_m_ext["updrs_z"] = X_ext[:, 3]

y_ext = df_m_ext["target_bin"].astype(int).values

print("Modeling N:", len(df_m_ext), "| fall rate:", y_ext.mean().round(3))

In [ ]:
# =========================================
# Cell Z — Bayesian logistic model EXTENDED
# Falls ~ Severity + Age + Duration + UPDRS
# =========================================

with pm.Model() as falls_model_ext:

    intercept = pm.Normal("intercept", 0, 1.5)

    beta_sev = pm.Normal("beta_sev", 0, 1)
    beta_age = pm.Normal("beta_age", 0, 1)
    beta_dur = pm.Normal("beta_dur", 0, 1)
    beta_updrs = pm.Normal("beta_updrs", 0, 1)

    logit_p = (
        intercept
        + beta_sev * df_m_ext["sev_z"].values
        + beta_age * df_m_ext["age_z"].values
        + beta_dur * df_m_ext["dur_z"].values
        + beta_updrs * df_m_ext["updrs_z"].values
    )

    p = pm.Deterministic("p_fall", pm.math.sigmoid(logit_p))

    pm.Bernoulli("falls", p=p, observed=y_ext)

    trace_falls_ext = pm.sample(
        draws=3000,
        tune=3000,
        chains=4,
        target_accept=0.95,
        random_seed=RANDOM_SEED
    )

az.summary(
    trace_falls_ext,
    var_names=["beta_sev", "beta_age", "beta_dur", "beta_updrs"],
    hdi_prob=0.95
)

In [ ]:
# =========================================
# Cell 14 — Bayesian logistic model WITH measurement error
# =========================================

with pm.Model() as falls_me_model:

    # -------------------------------
    # Priors
    # -------------------------------
    intercept = pm.Normal("intercept", 0, 1.5)
    beta_sev = pm.Normal("beta_sev", 0, 1)
    beta_age = pm.Normal("beta_age", 0, 1)
    beta_dur = pm.Normal("beta_dur", 0, 1)

    # -------------------------------
    # TRUE latent severity (unobserved)
    # Measurement error propagation
    # -------------------------------
    true_severity = pm.Normal(
        "true_severity",
        mu=df_m["sev_z"].values,
        sigma=df_m["latent_severity_sd"].values,
        shape=len(df_m)
    )

    # -------------------------------
    # Linear predictor
    # -------------------------------
    logit_p = (
        intercept
        + beta_sev * true_severity
        + beta_age * df_m["age_z"].values
        + beta_dur * df_m["dur_z"].values
    )

    p = pm.Deterministic("p_fall", pm.math.sigmoid(logit_p))

    # -------------------------------
    # Likelihood
    # -------------------------------
    pm.Bernoulli(
        "falls",
        p=p,
        observed=df_m["target_bin"].values
    )

    # -------------------------------
    # Sampling
    # -------------------------------
    trace_falls_me = pm.sample(
        draws=4000,
        tune=4000,
        chains=4,
        target_accept=0.95,
        random_seed=RANDOM_SEED
    )

# Summary
az.summary(
    trace_falls_me,
    var_names=["intercept", "beta_sev", "beta_age", "beta_dur"],
    hdi_prob=0.95
)

In [ ]:
beta_sev_samples = (
    trace_falls_me.posterior["beta_sev"]
    .stack(sample=("chain", "draw"))
    .values
)

p_beta_sev_pos = (beta_sev_samples > 0).mean()
print(f"P(beta_sev > 0) = {p_beta_sev_pos:.3f}")

In [ ]:
# =========================================
# Figure 3 / S3 — Latent severity & uncertainty
# =========================================

import matplotlib.pyplot as plt
import numpy as np

# Safety check
assert "latent_severity" in df_latent.columns
assert "latent_severity_sd" in df_latent.columns

sev = df_latent["latent_severity"].values
sev_sd = df_latent["latent_severity_sd"].values

plt.figure(figsize=(10, 4))

# -------------------------------
# Panel A — Latent severity
# -------------------------------
plt.subplot(1, 2, 1)
plt.hist(sev, bins=40, density=True, alpha=0.85)
plt.axvline(np.mean(sev), linestyle="--", linewidth=1)

plt.xlabel("Latent motor severity")
plt.ylabel("Density")
plt.title("A. Distribution of latent motor severity")

# -------------------------------
# Panel B — Severity uncertainty
# -------------------------------
plt.subplot(1, 2, 2)
plt.hist(sev_sd, bins=30, density=True, alpha=0.85)

plt.xlabel("Posterior SD of latent severity")
plt.ylabel("Density")
plt.title("B. Uncertainty of latent severity estimates")

plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# Figure X — Latent motor severity & uncertainty
# (npj Digital Medicine / Nature style)
# =========================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# -------------------------------
# Global style (Nature-like)
# -------------------------------
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# -------------------------------
# Safety checks
# -------------------------------
assert "latent_severity" in df_latent.columns
assert "latent_severity_sd" in df_latent.columns

sev = df_latent["latent_severity"].values
sev_sd = df_latent["latent_severity_sd"].values

# -------------------------------
# Figure canvas
# -------------------------------
fig, axes = plt.subplots(
    1, 2,
    figsize=(10.5, 4.2),
    gridspec_kw={"wspace": 0.35}
)

# ======================================================
# Panel A — Distribution of latent motor severity
# ======================================================
ax = axes[0]

# Histogram
ax.hist(
    sev,
    bins=35,
    density=True,
    color="#4C72B0",
    alpha=0.45,
    edgecolor="none"
)

# KDE
x_grid = np.linspace(sev.min() - 0.5, sev.max() + 0.5, 400)
kde = gaussian_kde(sev)
ax.plot(x_grid, kde(x_grid), color="black", linewidth=2)

# Median
median_sev = np.median(sev)
ax.axvline(
    median_sev,
    color="black",
    linestyle="--",
    linewidth=1.2
)

# 95% HDI (outline only)
hdi_low, hdi_high = np.percentile(sev, [2.5, 97.5])
ax.axvspan(
    hdi_low, hdi_high,
    facecolor="none",
    edgecolor="black",
    linestyle=":",
    linewidth=1.2
)

ax.set_title("A. Distribution of latent motor severity", loc="left")
ax.set_xlabel("Latent motor severity")
ax.set_ylabel("Density")

# ======================================================
# Panel B — Uncertainty of latent severity estimates
# ======================================================
ax = axes[1]

# Histogram (density but visually count-like avoided)
ax.hist(
    sev_sd,
    bins=30,
    density=True,
    color="#4C72B0",
    alpha=0.45,
    edgecolor="none"
)

# KDE
x_grid_sd = np.linspace(sev_sd.min() - 0.001, sev_sd.max() + 0.001, 400)
kde_sd = gaussian_kde(sev_sd)
ax.plot(x_grid_sd, kde_sd(x_grid_sd), color="black", linewidth=2)

ax.set_title("B. Uncertainty of latent severity estimates", loc="left")
ax.set_xlabel("Posterior SD of latent motor severity")
ax.set_ylabel("Density (a.u.)")

# -------------------------------
# Final layout
# -------------------------------
plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# Descriptive statistics for Results section
# =========================================

import numpy as np

print("\nLatent motor severity (posterior means)")
print("Mean:", np.mean(sev))
print("SD:", np.std(sev))
print("Median:", np.median(sev))
print("Min:", np.min(sev))
print("Max:", np.max(sev))
print("IQR:", np.percentile(sev, 75) - np.percentile(sev, 25))

print("\nPosterior uncertainty (posterior SD)")
print("Mean:", np.mean(sev_sd))
print("SD:", np.std(sev_sd))
print("Median:", np.median(sev_sd))
print("Min:", np.min(sev_sd))
print("Max:", np.max(sev_sd))
print("IQR:", np.percentile(sev_sd, 75) - np.percentile(sev_sd, 25))